# Extração de features — MobileNetV3 Large congelada

Para cada foto do dataset `vizwiz-224`, guardar a lista de números que a rede congelada produz (logo depois da média de cada mapa), junto com os rótulos e o split. Nada é treinado aqui: a regressão vem em outro passo.

## 1. Ambiente

Importar `torch` e `torchvision`, imprimir as versões e ver se a GPU está disponível (e qual é).

In [ ]:
# PyTorch: a biblioteca que faz as contas da rede neural (as somas ponderadas das camadas)
import torch
# torchvision: redes prontas e ferramentas para imagens; é de lá que vem a MobileNetV3
import torchvision


print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

# nome da GPU. Aqui a GPU só acelera a extração (uma passada por foto); a rede não é treinada.
print("GPU:", torch.cuda.get_device_name(0))

## 2. Manifesto

Ler o `manifest_224.csv` do dataset anexado (conferir o caminho em `/kaggle/input/`). Conferir quantas fotos há em cada split.

In [ ]:
import pandas as pd

# manifest = a tabela do dataset: uma linha por foto, com split, arquivo e os 4 rótulos (BLR, 
# BRT,DRK, FRM). Os rótulos são o gabarito dado por pessoas 
manifest = pd.read_csv("/kaggle/input/datasets/joyceximenes/vizwiz-224/manifest_224.csv")

# Validação = o conjunto que só MEDE: os pesos nunca são ajustados com ele 
s_val = (manifest.split == "val").sum()
s_train = (manifest.split == "train").sum()


print(f"quantidade de validação: {s_val}. de treino: {s_train}")

## 3. Modelo

Carregar a MobileNetV3 Large com os pesos pré-treinados, cortar a cabeça, congelar, passar para a GPU e deixar em modo de avaliação. Imprimir o nome dos pesos usados.

In [ ]:
# a MobileNetV3 Large e a classe com a lista dos pesos pré-treinados dela
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

# weights = os pesos que a rede aprendeu no pré-treino 
# (ImageNet, 1.000 classes de objetos), feito por outras pessoas. 
# É a parte "pré-treinada" da aula 
# DEFAULT é o mesmo que IMAGENET1K_V2
weights = MobileNet_V3_Large_Weights.DEFAULT

# cria a rede (filtros, média de cada mapa, cabeça) já com esses pesos carregados:
# são cerca de 5,5 milhões de números 
model = mobilenet_v3_large(weights=weights)

In [ ]:
# cortar a cabeça
# A "cabeça" (model.classifier) transforma a lista nos 1.000 scores do ImageNet.
# Identity() devolve a entrada sem mexer em nada: a saída da rede passa a ser a própria lista,
# a média de cada mapa. A regressão logística, com os 4 rótulos, vai ocupar o lugar da cabeça.
model.classifier = torch.nn.Identity()

In [ ]:
# Congelar os parâmetros
# model.parameters() = todos os pesos da rede (os milhões de números do pré-treino)
for param in model.parameters():
    # requires_grad = False: o PyTorch não calcula o gradiente desse peso,
    # então nenhum peso muda. É isso que "congelada" quer dizer 
    param.requires_grad = False

In [ ]:
# Definir dispositivo
# "cuda" é a GPU da NVIDIA; se não houver GPU, usa a CPU 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Enviar modelo para GPU/CPU
# move os pesos da rede para a memória da GPU; as fotos também terão de ir para lá 
model = model.to(device)

# Modo de avaliação
# eval() é diferente de congelar. Congelar impede o AJUSTE dos pesos. 
# eval() muda o COMPORTAMENTO de peças como o BatchNorm e o dropout, 
# para que o resultado de uma foto seja sempre o mesmo, sem depender das outras fotos do lote.
# Sem eval(), o BatchNorm continuaria atualizando estatísticas
# internas mesmo com os pesos congelados.
model.eval()

# Imprimir pesos utilizados
# registra no notebook qual versão dos pesos pré-treinados foi usada 
print("Pesos utilizados:", weights)
print("Dispositivo:", device)
# A partir daqui a rede só "aplica": entra uma foto, sai uma lista. 
# Nada é aprendido neste notebook

## 4. Dados

Ler cada PNG, passar a escala para 0 a 1 e normalizar com a média e o desvio do ImageNet. Montar os lotes.

In [ ]:
# foto vira um tensor para a rede
from PIL import Image
from torchvision import transforms
from torchvision.transforms import v2

# Pegar a primeira linha do manifest
linha = manifest.iloc[0]

# Montar o caminho completo da imagem
caminho = f"/kaggle/input/datasets/joyceximenes/vizwiz-224/{linha['arquivo']}"

# transforma o PNG em tensor
linha1 = torchvision.io.read_image(caminho)

# valor entre 0 e 1
tensor = linha1.float()/255

# normalizar com media e desvio de RBG
norm = torchvision.transforms.v2.Normalize(mean=[0.485, 0.456, 0.406], 
                                                 std=[0.229, 0.224, 0.225])

tensor_norm = norm(tensor)

print(f"tipo: {tensor_norm.dtype}")
print(f"tamanho: {tensor_norm.shape}")
print(f"mínimo: {tensor_norm.min()}")
print(f"máximo: {tensor_norm.max()}")


In [ ]:
# Peça B: o Dataset. Dataset e DataLoader são as duas peças padrão do PyTorch
# para alimentar a rede com muitas fotos
from torch.utils.data import Dataset, DataLoader

# pasta do dataset no Kaggle; a coluna "arquivo" do manifesto completa o caminho de cada foto
PASTA = "/kaggle/input/datasets/joyceximenes/vizwiz-224/"


# Um Dataset é uma "caixa" que responde a duas perguntas do PyTorch: 
# quantas fotos há? e me dê a foto de número i
class FotosVizWiz(Dataset):
    # guarda o que precisa para achar cada foto: a tabela (manifesto), 
    # a pasta e o normalizador
    def __init__(self, manifest, pasta, normalizador):
        # reset_index(drop=True) garante que a linha i da tabela é a i-ésima, a partir de 0
        self.manifest = manifest.reset_index(drop=True)
        self.pasta = pasta
        self.normalizador = normalizador

    # pergunta 1: quantas fotos há? (treino + validação = 31181)
    def __len__(self):
        return len(self.manifest)

    # pergunta 2: me dê a foto de número i. 
    def __getitem__(self, i):
        # nome do arquivo da foto i
        arquivo = self.manifest.loc[i, "arquivo"]
        # lê o PNG: inteiros de 0 a 255, forma 3 x 224 x 224 (aula: "A foto é uma grade de números")
        imagem = torchvision.io.read_image(self.pasta + arquivo)
        # passa para decimais de 0 a 1
        imagem = imagem.float() / 255
        # devolve a foto normalizada E o número i. O i serve para conferir, depois, que cada lista
        # extraída ficou na mesma ordem do manifesto e, portanto, junto do rótulo certo
        return self.normalizador(imagem), i

In [ ]:
# Peça C: o DataLoader agrupa as fotos em lotes (várias de uma vez, para a GPU render mais).
# É o jeito de fazer a extração de uma só vez, em lotes 
dataset = FotosVizWiz(manifest, PASTA, norm)

# batch_size=128: 128 fotos por vez (cabem folgadas na memória da T4).
# shuffle=False: NÃO embaralhar. As listas têm de sair na mesma ordem do manifesto, senão deixam de
# casar com os rótulos de cada linha, e o erro seria silencioso.
# num_workers=8: 8 processos leem e preparam as próximas fotos enquanto a GPU trabalha.

loader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=8)

# conferência: pega só o primeiro lote
lote, indices = next(iter(loader))
# esperado: 31181 fotos e 244 lotes (31181 / 128, arredondado para cima)
print("fotos no dataset:", len(dataset), "| lotes:", len(loader))
# esperado: torch.Size([128, 3, 224, 224]) e float32
print("lote:", lote.shape, lote.dtype)
# esperado: de 0 a 127
print("índices do lote: de", indices[0].item(), "a", indices[-1].item())
# a 1ª foto do lote tem de ser idêntica à que você normalizou na peça A (tensor_norm)
print("1ª foto igual à da peça A:", torch.equal(lote[0], tensor_norm))

## 5. Extração

Passar todas as fotos pela rede, sem calcular gradientes, e juntar as listas na mesma ordem do manifesto.

In [ ]:
# Etapa 5: a extração. Passa todos os lotes pela rede e junta as listas, na ordem do manifesto.
#  a rede só aplica o que já sabe, nada é aprendido
import time

listas = []  # uma matriz [fotos do lote, N] por lote
ordem = []  # os números i de cada lote, para conferir a ordem no fim (etapa 7)
inicio = time.time()

# torch.no_grad(): não guarda o que só serviria para calcular gradientes (para treinar).
# Nada aqui é treinado, então isso poupa memória e tempo
with torch.no_grad():
    # cada volta do for pega um lote: 128 fotos já normalizadas e os números i delas
    for k, (fotos, indices) in enumerate(loader, start=1):
        # manda o lote para a GPU, que é onde a rede está
        fotos = fotos.to(device)
        # a rede devolve a lista de cada foto (a média de cada mapa; a cabeça foi cortada): [128, N]
        saida = model(fotos)
        # traz o resultado de volta para a CPU e guarda; a GPU fica livre para o próximo lote
        listas.append(saida.cpu())
        ordem.append(indices)
        # só para acompanhar o andamento no log
        if k % 50 == 0 or k == len(loader):
            print(f"lote {k}/{len(loader)} | {time.time() - inicio:.0f} s", flush=True)

# junta os 244 pedaços em uma matriz só: uma linha por foto, na ordem do manifesto
features = torch.cat(listas)
ordem = torch.cat(ordem)

# N = quantos números tem a lista de cada foto = quantos mapas a 
# última camada de filtros produz
N = features.shape[1]
print("features:", features.shape, features.dtype)
print("N (números por foto):", N)
print(f"tempo da extração: {time.time() - inicio:.0f} s")

## 6. Guardar

Salvar as listas e o manifesto, na mesma ordem, em `/kaggle/working/`.

In [ ]:
# Etapa 6: guardar. Salva as listas e o manifesto, na mesma ordem, em /kaggle/working/ 
# Sem isso, as listas se perdem quando a execução termina
import json
import os

import numpy as np

SAIDA = "/kaggle/working/"

# as listas viram uma matriz NumPy (31181 x 960, float32). .numpy() converte o tensor da CPU
np.save(SAIDA + "features_large.npy", features.numpy())

# o manifesto na mesma ordem. A coluna "i" é a posição de cada foto, para casar cada linha da
# matriz com o split e os 4 rótulos dela (os rótulos são o gabarito; as listas, as features)
manifesto_saida = manifest.reset_index(drop=True).copy()
manifesto_saida.insert(0, "i", range(len(manifesto_saida)))
manifesto_saida.to_csv(SAIDA + "features_manifest.csv", index=False)

# o que é preciso saber para reproduzir e citar: a rede, os pesos, as versões e a normalização
info = {
    "modelo": "mobilenet_v3_large (classifier trocado por Identity)",
    "pesos": str(weights),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "N": int(N),
    "fotos": int(len(manifesto_saida)),
    "tamanho_da_imagem": 224,
    "normalizacao": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
}
with open(SAIDA + "features_info.json", "w", encoding="utf-8") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

# o que ficou gravado, com o tamanho de cada arquivo
for nome in sorted(os.listdir(SAIDA)):
    print(f"{nome}: {os.path.getsize(SAIDA + nome) / 1024**2:.1f} MB")

## 7. Conferências

Forma da matriz, valores inválidos, ordem, fotos por split e 8 fotos passadas uma a uma comparadas com as do lote.

In [ ]:
# Etapa 7: conferências. Cada uma imprime o resultado e, no fim, um aviso geral. São a prova de que
# as listas estão certas e na ordem certa antes de a regressão usá-las
resultados = {}

# 1. forma: uma linha por foto e 960 colunas
resultados["forma"] = tuple(features.shape) == (31181, 960)
print("1. forma:", tuple(features.shape), "| esperado (31181, 960):", resultados["forma"])

# 2. valores inválidos: NaN (não é número) ou infinito deixariam a regressão sem sentido
resultados["validos"] = bool(torch.isfinite(features).all())
print("2. sem NaN nem infinito:", resultados["validos"])

# 3. ordem: o número i de cada foto, na ordem em que saiu dos lotes, tem de ser 0, 1, 2, ... sem
# pular nem repetir. Por isso o DataLoader não embaralha e o Dataset devolve o i
resultados["ordem"] = torch.equal(ordem, torch.arange(len(manifest)))
print("3. ordem igual à do manifesto:", resultados["ordem"])

# 4. fotos por split
contagem = manifest.split.value_counts().to_dict()
resultados["splits"] = contagem == {"train": 23431, "val": 7750}
print("4. fotos por split:", contagem, "| esperado train 23431 e val 7750:", resultados["splits"])

# 5. 8 fotos passadas uma a uma x as mesmas no lote. Sorteio fixo: as 4 primeiras, uma do meio, a
# última de treino, a primeira e a última de validação (as fronteiras são onde um erro de ordem aparece)
primeira_val = int((manifest.split == "val").idxmax())
amostra = [0, 1, 2, 3, 15000, primeira_val - 1, primeira_val, len(manifest) - 1]
with torch.no_grad():
    sozinhas = torch.cat([model(dataset[i][0].unsqueeze(0).to(device)).cpu() for i in amostra])
diferenca = (sozinhas - features[amostra]).abs().max().item()
# a conta feita com uma foto ou com 128 pode diferir nas últimas casas do float32; 1e-3 é folgado
resultados["lote_x_sozinha"] = diferenca < 1e-3
print("5. maior diferença entre foto sozinha e no lote:", diferenca, "| menor que 1e-3:", resultados["lote_x_sozinha"])

# 6. o arquivo gravado, lido de volta, é igual à matriz em memória
lida = np.load(SAIDA + "features_large.npy")
resultados["arquivo"] = np.array_equal(lida, features.numpy())
print("6. arquivo salvo igual à matriz em memória:", resultados["arquivo"])

print()
print("TUDO CERTO" if all(resultados.values()) else "ATENÇÃO, algo falhou: " + str([k for k, v in resultados.items() if not v]))

## Resumo: o que este notebook entrega

- `features_large.npy`: matriz de 31181 linhas por 960 colunas (`float32`). Cada linha é a lista de uma foto: a média de cada um dos 960 mapas da última camada de filtros da MobileNetV3 Large, sem a cabeça.
- `features_manifest.csv`: o manifesto na mesma ordem das linhas da matriz. A coluna `i` é a posição da foto, e o arquivo traz também o split e os 4 rótulos (BLR, BRT, DRK, FRM).
- `features_info.json`: o nome dos pesos, as versões do PyTorch e do torchvision, o N e a normalização usada, para reproduzir e citar no artigo.

Nada é treinado aqui. O próximo passo é ajustar uma regressão logística por rótulo nas linhas de treino e medir nas de validação.